# 06 - Verify the station reference data

The Oracle hub stores, for each station, the BFS municipality number, the canton and LV95 coordinates. This notebook checks them against the official commune boundaries of swisstopo (swissBOUNDARIES3D): for each station, the commune containing its LV95 point is retrieved and compared with the hub.

| Hub value | Official value | Meaning of a mismatch |
|---|---|---|
| `municipality_bfs_nr` | BFS number of the commune containing the point | wrong number, outdated number after a merger, or wrong coordinates |
| `canton_code` | canton of that commune | idem |

A mismatch is resolved on the map: if the point lies on the station, the coordinates are right and the hub value is wrong.

In [1]:
import time

import oracledb
import pandas as pd
import requests

from ingestion.stations import connection_settings

IDENTIFY_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/identify"
COMMUNES_LAYER = "ch.swisstopo.swissboundaries3d-gemeinde-flaeche.fill"

pd.set_option("display.width", 160)

## Stations from the hub

Read from the hub selected by `ORACLE_TARGET` (local container or Oracle Cloud), as in the ingestion.

In [2]:
target, connect_kwargs = connection_settings()
query = """
    SELECT station_id, station_name, municipality_bfs_nr, canton_code, lv95_e, lv95_n
    FROM stations
    ORDER BY station_id
"""
with oracledb.connect(**connect_kwargs) as conn:
    hub = pd.read_sql(query, conn)
hub.columns = hub.columns.str.lower()
print(f"{len(hub)} stations from oracle:{target}")
hub

16 stations from oracle:local


/tmp/ipykernel_15722/3300997197.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  hub = pd.read_sql(query, conn)


,station_id,station_name,municipality_bfs_nr,canton_code,lv95_e,lv95_n
0,8500010,Basel SBB,2701,BS,2611363.00,1266308.94
1,8500109,Delémont,6711,JU,2593310.00,1245689.06
2,8500218,Olten,2581,SO,2635441.47,1244669.72
3,8501008,Genève,6621,GE,2499967.85,1118469.78
4,8501120,Lausanne,5586,VD,2537874.64,1152042.74
5,8501506,Sion,6266,VS,2593869.35,1119569.86
6,8503000,Zürich HB,261,ZH,2683212.09,1248029.61
7,8504100,Fribourg/Freiburg,2196,FR,2578046.18,1183594.04
8,8504221,Neuchâtel,6458,NE,2561746.61,1205197.48
9,8504300,Biel/Bienne,371,BE,2585151.73,1220231.08


## Commune containing each point

`identify` returns the features of a layer at a point. On the commune layer, the feature id is the BFS municipality number.

In [3]:
def commune_at(east, north):
    params = {
        "geometryType": "esriGeometryPoint",
        "geometry": f"{east},{north}",
        "sr": 2056,
        "layers": f"all:{COMMUNES_LAYER}",
        "tolerance": 0,
        "returnGeometry": "false",
    }
    response = requests.get(IDENTIFY_URL, params=params, timeout=15,
                            headers={"User-Agent": "swiss-mobility-data-platform/reference-check"})
    response.raise_for_status()
    return response.json()["results"]


# Structure of one answer, to confirm the attribute names before using them
sample = commune_at(hub.loc[0, "lv95_e"], hub.loc[0, "lv95_n"])
sample[0] if sample else "no commune found at this point"

{'layerBodId': 'ch.swisstopo.swissboundaries3d-gemeinde-flaeche.fill',
 'layerName': 'Gemeindegrenzen',
 'featureId': '16358-1850',
 'id': '16358-1850',
 'attributes': {'gemname': 'Basel',
  'gemflaeche': 2232.522916592845,
  'gde_hist_id': 16358,
  'gde_nr': 7700,
  'jahr': 1850,
  'perimeter': 26039.01953612537,
  'kanton': 'BS',
  'objektart': 11,
  'is_current_jahr': False,
  'objektart_lookup': 'politische_gemeinde',
  'link_de': 'https://www.agvchapp.bfs.admin.ch/de/communes/results?BfsNr=7700&EntriesFrom=12.09.1848&EntriesTo=31.12.1907&IncludeUnassignedEntities=True',
  'link_fr': 'https://www.agvchapp.bfs.admin.ch/fr/communes/results?BfsNr=7700&EntriesFrom=12.09.1848&EntriesTo=31.12.1907&IncludeUnassignedEntities=True',
  'link_it': 'https://www.agvchapp.bfs.admin.ch/it/communes/results?BfsNr=7700&EntriesFrom=12.09.1848&EntriesTo=31.12.1907&IncludeUnassignedEntities=True',
  'label': 'Basel'}}

In [5]:
def current_commune(results):
    """The current version of the commune among the historical ones returned by the layer."""
    current = [r for r in results if r.get("attributes", {}).get("is_current_jahr")]
    if current:
        return current[0]
    # Fallback: the most recent year available
    return max(results, key=lambda r: r.get("attributes", {}).get("jahr", 0)) if results else None


rows = []
for station in hub.itertuples():
    feature = current_commune(commune_at(station.lv95_e, station.lv95_n))
    attributes = feature.get("attributes", {}) if feature else {}
    rows.append({
        "station_id": station.station_id,
        "official_bfs_nr": attributes.get("gde_nr"),
        "official_name": attributes.get("gemname"),
        "official_canton": attributes.get("kanton"),
        "official_year": attributes.get("jahr"),
    })
    time.sleep(0.2)    # public service

official = pd.DataFrame(rows).astype({"official_bfs_nr": "Int64"})

## Comparison

In [6]:
check = hub.merge(official, on="station_id")
check["bfs_ok"] = (check["municipality_bfs_nr"] == check["official_bfs_nr"]).fillna(False).astype(bool)
check["canton_ok"] = (check["canton_code"] == check["official_canton"]).fillna(False).astype(bool)
check["map"] = ("https://map.geo.admin.ch/?E=" + check["lv95_e"].round().astype(int).astype(str)
                + "&N=" + check["lv95_n"].round().astype(int).astype(str) + "&zoom=10&crosshair=marker")

print(f"BFS number matches: {check['bfs_ok'].sum()} of {len(check)}")
print(f"Canton matches:     {check['canton_ok'].sum()} of {len(check)}")
check[["station_name", "municipality_bfs_nr", "official_bfs_nr", "official_name",
       "canton_code", "official_canton", "bfs_ok", "canton_ok"]]

BFS number matches: 16 of 16
Canton matches:     16 of 16


,station_name,municipality_bfs_nr,official_bfs_nr,official_name,canton_code,official_canton,bfs_ok,canton_ok
0,Basel SBB,2701,2701,Basel,BS,BS,True,True
1,Delémont,6711,6711,Delémont,JU,JU,True,True
2,Olten,2581,2581,Olten,SO,SO,True,True
3,Genève,6621,6621,Genève,GE,GE,True,True
4,Lausanne,5586,5586,Lausanne,VD,VD,True,True
5,Sion,6266,6266,Sion,VS,VS,True,True
6,Zürich HB,261,261,Zürich,ZH,ZH,True,True
7,Fribourg/Freiburg,2196,2196,Fribourg,FR,FR,True,True
8,Neuchâtel,6458,6458,Neuchâtel,NE,NE,True,True
9,Biel/Bienne,371,371,Biel/Bienne,BE,BE,True,True


In [7]:
# Mismatches, with a map link centred on the stored coordinates
mismatches = check[~(check["bfs_ok"] & check["canton_ok"])]
for row in mismatches.itertuples():
    print(f"{row.station_name}: hub {row.municipality_bfs_nr}/{row.canton_code}, "
          f"official {row.official_bfs_nr} {row.official_name}/{row.official_canton}")
    print(f"  {row.map}")
if mismatches.empty:
    print("No mismatch")

No mismatch


## Observations

- The 16 stations of the hub were checked against the current commune boundaries of swissBOUNDARIES3D (`is_current_jahr = True`): BFS municipality number and canton match for 16 of 16 stations.
- Each LV95 point falls inside the expected commune, which also validates the coordinates converted from WGS84.
- The commune layer contains every historical version of the boundaries since 1850: a point returns several features (e.g. Basel, 1850, BFS 7700). Only the current version is used; the BFS number is the attribute `gde_nr`, not the feature id.
- Neuchâtel carries the number assigned after the 2021 merger (6458).